In [ ]:
from pathlib import Path

import numpy as np
import seaborn as sns
from bonner.plotting import save_figure
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

from lib.datasets import (
    compute_shared_stimuli,
    filter_by_stimulus,
    nsd,
    split_by_repetition,
)
from lib.spectra import (
    compute_within_individual_spectra,
    plot_spectra,
)
from lib.utilities import JOURNAL_MATPLOTLIBRC, mathtext_exponent_label

FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

REFERENCE_SUBJECT = 0

In [ ]:
datasets = {"anterior": {}, "posterior": {}}

for subject in tqdm(range(nsd.N_SUBJECTS), desc="subject", leave=False):
    dataset = nsd.load_dataset(
        subject=subject,
        roi="general",
        preprocessing="fithrf",
        z_score=True,
    ).load()

    stimuli = compute_shared_stimuli([dataset], n_repetitions=2)

    median = np.median(dataset["y"].to_numpy())
    dataset = dataset.sortby("y")
    n_half = dataset.sizes["neuroid"] // 2
    datasets["posterior"][subject] = split_by_repetition(
        filter_by_stimulus(
            dataset.isel(neuroid=range(n_half)).rename(
                f"{dataset.name}.posterior",
            ),
            stimuli=stimuli,
        ),
        n_repetitions=2,
    )
    datasets["anterior"][subject] = split_by_repetition(
        filter_by_stimulus(
            dataset.isel(neuroid=range(n_half, dataset.sizes["neuroid"])).rename(
                f"{dataset.name}.anterior",
            ),
            stimuli=stimuli,
        ),
        n_repetitions=2,
    )

In [ ]:
spectra_within = {
    roi: compute_within_individual_spectra(datasets_, n_permutations=5_000)
    for roi, datasets_ in tqdm(datasets.items(), desc="region of interest", leave=False)
}

In [ ]:
fig, axes = plt.subplots(figsize=(4, 3), ncols=2, sharex=True, sharey=True)

for (roi, spectra_), ax in zip(
    tqdm(spectra_within.items(), desc="region of interest", leave=False),
    axes.flat,
    strict=True,
):
    plot_spectra(
        spectra=spectra_,
        ax=ax,
        palette="crest",
        hue="individual",
        hue_order=list(reversed(range(nsd.N_SUBJECTS))),
        hue_labels=[f"{1 + subject}" for subject in reversed(range(nsd.N_SUBJECTS))],
        marker="s",
        hide_insignificant=True,
        null_quantile=0.999,
    )
    ax.set_title(roi)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(left=1, right=1e4)
    ax.set_xticks([1, 1e1, 1e2, 1e3, 1e4])
    ax.set_ylim(bottom=1e-8, top=1e-1)

    ytick_exponents = list(range(-8, -1))
    ax.set_yticks(
        [10**exponent for exponent in ytick_exponents],
        labels=[
            mathtext_exponent_label(exponent) if exponent % 2 == 0 else ""
            for exponent in ytick_exponents
        ],
    )

_ = axes[-1].legend(
    loc="upper right",
    title="subject",
    ncols=2,
    columnspacing=0.5,
    handletextpad=0.0,
    borderpad=0,
    borderaxespad=0,
    reverse=True,
)
fig.supxlabel("rank", y=0.075, x=0.55)
fig.suptitle("within-subject spectra,\ngeneral visually responsive region")
axes[0].set_ylabel("covariance")

save_figure(fig, filepath=FIGURES_HOME / "anterior-vs-posterior.pdf")